# Treino do detector de EPI (capacete e óculos) — YOLO26n

Treina a partir do dataset montado **no seu computador** por `scripts/filter_dataset.py`
(Roboflow + sintéticas + teste real), enviado ao Google Drive como zip.

**Antes de rodar:**
1. No PowerShell, na pasta do projeto: `Compress-Archive data\processed\epi\* epi_dataset.zip`
2. Suba o `epi_dataset.zip` para a raiz do seu Google Drive ("Meu Drive").
3. No Colab: **Ambiente de execução → Alterar o tipo de ambiente → GPU T4**.

Os resultados são salvos no próprio Drive: se a sessão cair, nada se perde.

In [ ]:
!nvidia-smi
!pip install -q ultralytics

## 1. Conectar o Google Drive e extrair o dataset

In [ ]:
import glob, os, shutil, zipfile
from google.colab import drive

drive.mount("/content/drive")

DATASET_ZIP = "/content/drive/MyDrive/epi_dataset.zip"   # ajuste se salvou em outra pasta
DATA_DIR = "/content/datasets/epi"
RUNS_DIR = "/content/drive/MyDrive/epi_runs"              # resultados ficam no Drive
RUN_NAME = "epi_yolo26n_v2"

def extrair(zip_path, destino):
    """Extrai o zip convertendo '\\' em '/': o Compress-Archive do Windows grava
    os caminhos com barra invertida, que o Linux não entende como pastas."""
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            nome = info.filename.replace("\\", "/")
            alvo = os.path.join(destino, nome)
            if nome.endswith("/"):
                os.makedirs(alvo, exist_ok=True)
                continue
            os.makedirs(os.path.dirname(alvo), exist_ok=True)
            with z.open(info) as src, open(alvo, "wb") as dst:
                shutil.copyfileobj(src, dst)

# só reaproveita uma extração anterior se ela estiver completa
if not glob.glob(f"{DATA_DIR}/**/valid/images", recursive=True):
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    os.makedirs(DATA_DIR)
    print("Extraindo o dataset (alguns minutos)...")
    extrair(DATASET_ZIP, DATA_DIR)
# acha o data.yaml mesmo se o zip tiver uma pasta a mais dentro
DATA = sorted(glob.glob(f"{DATA_DIR}/**/data.yaml", recursive=True), key=len)[0]
DATA_REAL = DATA.replace("data.yaml", "data_teste_real.yaml")
print("data.yaml:", DATA)
print("teste real:", DATA_REAL if os.path.isfile(DATA_REAL) else "não incluído")
for split in ["train", "valid", "test", "test_real"]:
    d = os.path.join(os.path.dirname(DATA), split, "images")
    if os.path.isdir(d):
        print(f"{split:>9}: {len(os.listdir(d))} imagens")
    elif split != "test_real":
        raise SystemExit(f"Pasta {d} não encontrada: confira o conteúdo do zip")

## 2. Treinar
Se a sessão cair no meio, rode esta célula de novo: ela retoma do último checkpoint salvo no Drive.

In [ ]:
from ultralytics import YOLO

last = f"{RUNS_DIR}/{RUN_NAME}/weights/last.pt"
if os.path.isfile(last):
    print("Retomando treino interrompido...")
    model = YOLO(last)
    model.train(resume=True)
else:
    model = YOLO("yolo26n.pt")
    model.train(
        data=DATA,
        epochs=50,
        imgsz=640,
        batch=16,
        patience=15,
        project=RUNS_DIR,
        name=RUN_NAME,
        exist_ok=True,
    )
SAVE_DIR = f"{RUNS_DIR}/{RUN_NAME}"
print("Resultados em:", SAVE_DIR)

## 3. Avaliar
Duas avaliações separadas: o teste do Roboflow e as fotos reais da webcam (se incluídas). A segunda é a que mede o uso real.

In [ ]:
from ultralytics import YOLO

SAVE_DIR = f"{RUNS_DIR}/{RUN_NAME}"
best = YOLO(f"{SAVE_DIR}/weights/best.pt")

def avaliar(yaml_path, titulo, nome):
    m = best.val(data=yaml_path, split="test", project=SAVE_DIR, name=nome, exist_ok=True)
    print(f"\n=== {titulo} ===")
    print(f"mAP50: {m.box.map50:.3f} | mAP50-95: {m.box.map:.3f} | P: {m.box.mp:.3f} | R: {m.box.mr:.3f}")
    for i, name in best.names.items():
        print(f"{name:>11}: mAP50-95 = {m.box.maps[i]:.3f}")

avaliar(DATA, "Teste Roboflow", "teste_roboflow")
if os.path.isfile(DATA_REAL):
    avaliar(DATA_REAL, "Teste REAL (webcam)", "teste_real")

## 4. Ver algumas predições

In [ ]:
from IPython.display import Image, display

base = os.path.dirname(DATA)
imgs = sorted(glob.glob(f"{base}/test_real/images/*"))[:4] + sorted(glob.glob(f"{base}/test/images/*"))[:4]
best.predict(imgs, conf=0.45, save=True, project=SAVE_DIR, name="preview", exist_ok=True)
for p in sorted(glob.glob(f"{SAVE_DIR}/preview/*.jpg"))[:8]:
    display(Image(filename=p, width=450))

## 5. Baixar os resultados
Os arquivos já estão no Drive em `epi_runs/`; esta célula só gera um zip para baixar direto.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/resultados_treino_v2", "zip", SAVE_DIR)
files.download("/content/resultados_treino_v2.zip")